# Qwen2.5-7B — VHE QLoRA with DEV Early Stopping

Notebook huấn luyện Qwen2.5-7B trên dữ liệu Event Extraction theo định dạng `input_text` → `target_text`, đánh giá toàn bộ DEV sau mỗi epoch và early stopping.

**Các phần có thể tùy chỉnh khi chạy:**
- `model_path`: đường dẫn model nền.
- Đường dẫn `train.jsonl` và `dev.jsonl`: dataset.
- `CHECKPOINT` và `RESUME_FROM_CHECKPOINT`: chọn train từ đầu hoặc tiếp tục từ checkpoint.

> Checkpoint tốt nhất được chọn theo `eval_avg_f1`; TEST không được dùng trong quá trình chọn model.

# Link resourse Kaggle (model, dataset, notebook)
## Model
- Qwen2.5-7B: https://www.kaggle.com/models/qwen-lm/qwen2.5/Transformers/7b

## Dataset
- GENEVA: https://www.kaggle.com/datasets/md0612/geneva-adapted
- VHE: https://www.kaggle.com/datasets/md0612/vhe-adapted
- BKEE: https://www.kaggle.com/datasets/md0612/bkee-adapted
- PHEE: 

## Metrics
- https://www.kaggle.com/datasets/md0612/metrics

## Sample notebook này
- 

## 1. Cài đặt thư viện

Cài hoặc nâng cấp `bitsandbytes` để hỗ trợ tải model ở chế độ 4-bit.

In [ ]:
!pip install -U "bitsandbytes>=0.46.1"

## 2. Import thư viện

Import các thư viện dùng cho load model, xử lý dataset, QLoRA và training.

In [ ]:
from pathlib import Path
import json
import gc
import sys
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    EarlyStoppingCallback,
)
from peft import (
    prepare_model_for_kbit_training,
    get_peft_model,
    LoraConfig,
)

## 3. Load tokenizer và base model

Load Qwen2.5-7B đã gắn vào Kaggle Notebook và cấu hình QLoRA 4-bit cho GPU T4.

In [ ]:
model_path = Path("/kaggle/input/qwen2.5/transformers/7b/1")
if not model_path.exists():
    raise FileNotFoundError(
        f"Không tìm thấy Qwen2.5-7B tại {model_path}. "
        "Hãy Add Model Qwen2.5-7B vào Kaggle Notebook."
    )

# 4-bit quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, # Load 4-bit, giảm VRAM
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(model_path) # Load tokenizer

# Load model
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    quantization_config=bnb_config,
    device_map="auto",
)

# Chuẩn bị model cho QLoRA
model = prepare_model_for_kbit_training(model)

# Quan trọng khi dùng gradient checkpointing
model.config.use_cache = False
model.gradient_checkpointing_enable()

print("Model loaded!")
print("Device:", model.device)

## 4. Tokenize một example

Ghép token của input và target; chỉ tính loss trên phần target và EOS.

In [ ]:
def tokenize_example(example):
    input_text = example["input_text"]
    target_text = example["target_text"]

    input_tokens = tokenizer(
        input_text,
        add_special_tokens=True,
    )

    target_tokens = tokenizer(
        target_text,
        add_special_tokens=False,
    )

    input_ids = input_tokens["input_ids"]
    target_ids = target_tokens["input_ids"]

    input_ids = (
        input_tokens["input_ids"]
        + target_ids
        + [tokenizer.eos_token_id]
    ) # Thêm EOS để model dừng lại

    labels = (
        [-100] * len(input_tokens["input_ids"])
        + target_ids
        + [tokenizer.eos_token_id]
    ) # Thêm EOS để model dừng lại khi generate

    return {
        "input_ids": input_ids,
        "labels": labels,
        "attention_mask": [1] * len(input_ids),
    }

## 5. Format dữ liệu Event Extraction

Chuyển mỗi sample từ `text` và `events` sang cặp `input_text` / `target_text`.

In [ ]:
def format_example(example):
    text = example["text"]

    target = {
        "events": example["events"]
    }

    return {
        "input_text": text,
        "target_text": json.dumps(
            target,
            ensure_ascii=False,
            separators=(",", ":")
        )
    }

## 6. Data collator

Pad các sample trong batch; phần padding của `labels` được gán `-100` để không tính loss.

In [ ]:
def ee_data_collator(features):
    max_length = max(len(f["input_ids"]) for f in features)

    input_ids = []
    attention_mask = []
    labels = []

    for f in features:
        pad_length = max_length - len(f["input_ids"])

        input_ids.append(
            f["input_ids"] + [tokenizer.pad_token_id] * pad_length
        )

        attention_mask.append(
            f["attention_mask"] + [0] * pad_length
        )

        labels.append(
            f["labels"] + [-100] * pad_length
        )

    return {
        "input_ids": torch.tensor(input_ids, dtype=torch.long),
        "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
        "labels": torch.tensor(labels, dtype=torch.long),
    }

## 7. Cấu hình và áp dụng LoRA

Khai báo cấu hình LoRA và gắn adapter vào model đã được chuẩn bị cho k-bit training.

In [ ]:
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)

model = get_peft_model(model, peft_config)

## 8. Load và preprocess dataset

Đọc train/dev JSONL, format dữ liệu rồi tokenize. Thay hai đường dẫn nếu muốn dùng dataset khác.

In [ ]:
# dataset = load_dataset(
#     "json",
#     data_files={
#         "train": "## Data path/train.jsonl",
#         "dev": "## Data path/dev.jsonl",
#     },
# )

dataset = load_dataset(
    "json",
    data_files={
        "train": "/kaggle/input/datasets/md0612/vhe-adapted/train.jsonl",
        "dev": "/kaggle/input/datasets/md0612/vhe-adapted/dev.jsonl",
    },
)

# Raw datasets
train_raw = dataset["train"]
dev_raw = dataset["dev"]

# Formatted datasets
train_formatted = train_raw.map(format_example)
dev_formatted = dev_raw.map(format_example)

# Tokenized datasets for training
train_dataset = train_formatted.map(
    tokenize_example,
    remove_columns=train_formatted.column_names,
)

# Tokenized dataset for validation loss (nếu sử dụng)
dev_dataset = dev_formatted.map(
    tokenize_example,
    remove_columns=dev_formatted.column_names,
)

# Raw/formatted dataset for generation evaluation
dev_generation_dataset = dev_formatted

## 9. DEV metrics

Sau mỗi epoch, model sinh kết quả trên toàn bộ DEV rồi tính TI, TC, AI, AC và `avg_f1`.

In [ ]:
# sys.path.insert(0, "## Metrics path")

sys.path.insert(0, "/kaggle/input/datasets/md0612/metrics")

from metrics.evaluator import Evaluator

metric_calculator = Evaluator()

## Load Custom Trainer & Evaluator

Import custom GenerationTrainer and Evaluator from the project source.

- GenerationTrainer: Performs autoregressive generation during validation.
- Evaluator: Computes TI, TC, AI, AC and average F1.

In [ ]:
import sys

sys.path.insert(0, "/kaggle/input/datasets/md0612/ee-source")

from trainer.generation_trainer import GenerationTrainer

## 10. TrainingArguments

Đánh giá và lưu checkpoint sau mỗi epoch; chọn checkpoint có `eval_avg_f1` cao nhất.

In [ ]:
training_args = TrainingArguments(
    output_dir="/kaggle/working/ee_qwen2_5_7b/QLORA",

    num_train_epochs=10,

    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,

    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=4,

    load_best_model_at_end=True,
    metric_for_best_model="avg_f1",
    greater_is_better=True,

    gradient_checkpointing=True,
    fp16=True,
    report_to="none",
)


Early stopping dùng toàn bộ DEV, patience 3 epoch và ngưỡng cải thiện 0.001.

In [ ]:
EARLY_STOPPING_PATIENCE = 3
EARLY_STOPPING_THRESHOLD = 0.001

## 11. Giải phóng bộ nhớ và kiểm tra GPU

Dọn garbage collection/cache CUDA rồi in mức memory đang dùng trên từng GPU.

In [ ]:
gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect()

for i in range(torch.cuda.device_count()):
    print(
        f"GPU {i}: "
        f"{torch.cuda.memory_allocated(i) / 1024**3:.2f} GB allocated, "
        f"{torch.cuda.memory_reserved(i) / 1024**3:.2f} GB reserved"
    )

## 12. Khởi tạo Trainer và kiểm tra batch

Khởi tạo `GenerationTrainer` với TRAIN và toàn bộ DEV; kiểm tra một batch DEV trước khi train.

In [ ]:
trainer = GenerationTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=dev_dataset,
    generation_dataset=dev_generation_dataset,
    tokenizer=tokenizer,
    evaluator=metric_calculator,
    max_new_tokens=512,
    data_collator=ee_data_collator,
    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=EARLY_STOPPING_PATIENCE,
            early_stopping_threshold=EARLY_STOPPING_THRESHOLD,
        )
    ],
)

batch = next(iter(trainer.get_eval_dataloader()))
print("DEV samples:", len(dev_dataset))
print("DEV batch shape:", tuple(batch["input_ids"].shape))

DEV generation và early stopping đã được bật trong trainer phía trên.

In [ ]:
print(
    "Early stopping:",
    f"patience={EARLY_STOPPING_PATIENCE},",
    f"threshold={EARLY_STOPPING_THRESHOLD}",
)

## 13. Chọn chế độ train: từ đầu hoặc resume

- Đặt `RESUME_FROM_CHECKPOINT = None` để train từ đầu.
- Đặt `RESUME_FROM_CHECKPOINT` bằng đường dẫn checkpoint để tiếp tục training.

Đường dẫn checkpoint hiện tại được giữ theo notebook gốc. Khi đổi model/dataset hoặc checkpoint, cập nhật các đường dẫn tương ứng ở những cell phía trên và cell này.

In [ ]:
# CHECKPOINT = "/kaggle/input/models/md0612/qwen3-4b-qlora-bkee-all-checkpoint/pytorch/default/4/checkpoint-8176"

# Train từ đầu: RESUME_FROM_CHECKPOINT = None
# Resume: RESUME_FROM_CHECKPOINT = CHECKPOINT
RESUME_FROM_CHECKPOINT = None

trainer.train(
    resume_from_checkpoint=RESUME_FROM_CHECKPOINT
)

## 14. Lưu model và tokenizer cuối cùng

Lưu adapter/model mà `Trainer` quản lý cùng tokenizer vào thư mục output.

In [ ]:
BEST_MODEL_DIR = "/kaggle/working/ee_qwen2_5_7b/best"
trainer.save_model(BEST_MODEL_DIR)
tokenizer.save_pretrained(BEST_MODEL_DIR)

print("Best checkpoint:", trainer.state.best_model_checkpoint)
print("Best DEV avg_f1:", trainer.state.best_metric)
print("Saved best model to:", BEST_MODEL_DIR)